# Phase 1: Decoding Ablation (Base vs Pilot 1 vs Pilot 2)

Evaluates **base**, **pilot_01**, and **pilot_02** under controlled repetition-penalty decoding (`do_sample=False`, `max_new_tokens=2048`, `float32`) on the frozen 10-case validation set (`900825a06d38a006`), using the exact `evaluate_output` metrics from `compare_pilot_clinical.ipynb`.

All existing datasets, baseline outputs, and `pilot_01`/`pilot_02` comparison reports are preserved untouched. Results are saved under a new explicit protocol directory: `decoding_ablation_10/`.

In [ ]:
%pip install -q --progress-bar on "transformers==4.57.6" "datasets>=3,<5" huggingface_hub pandas accelerate "peft==0.18.1"

In [ ]:
import os, json, hashlib, random, re, time, zipfile, importlib.metadata
from pathlib import Path
from getpass import getpass
from collections import Counter
import pandas as pd
import torch
from datasets import load_from_disk
from huggingface_hub import get_token
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor, LogitsProcessorList
from peft import PeftModel

def digest(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True).encode()).hexdigest()

def file_sha(path):
    sha = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            sha.update(block)
    return sha.hexdigest()

def read_token():
    token = os.environ.get("HF_TOKEN") or get_token()
    if not token:
        print("Enter your Hugging Face token in the input prompt.", flush=True)
        token = getpass("Hugging Face read token: ").strip()
    if not token:
        raise RuntimeError("A Hugging Face token with accepted Gemma access is required.")
    os.environ["HF_TOKEN"] = token
    return token

# Optional convenience: unzip bundled workspace data if uploaded to /content/colab_ablation_bundle.zip
bundle_zip = Path("/content/colab_ablation_bundle.zip")
if bundle_zip.is_file() and not Path("/content/prepared_clinical_dataset/latest.json").is_file():
    print("Extracting", bundle_zip, "to /content ...", flush=True)
    with zipfile.ZipFile(bundle_zip, "r") as zf:
        zf.extractall("/content")
    print("Extraction complete.", flush=True)

def runtime_path(path, sentinel):
    path = Path(path).expanduser()
    candidates = [path]
    if not path.is_absolute():
        candidates += [parent / path for parent in Path.cwd().parents]
        candidates.append(Path("/content") / path)
    for candidate in candidates:
        if (candidate / sentinel).is_file():
            return candidate.resolve()
    raise FileNotFoundError(f"Kernel directory: {Path.cwd()}; missing {sentinel} under {path}.")

PREPARED_ROOT = runtime_path("prepared_clinical_dataset", "latest.json")
BASELINE_DIR = runtime_path("baseline_evaluation_diagnostic_10/900825a06d38a006_353c8393d011", "comparison_contract.json")
PILOT_01_DIR = runtime_path("pilot_01", "adapter/adapter_config.json")
PILOT_02_DIR = runtime_path("pilot_02", "adapter/adapter_config.json")
OUTPUT_ROOT = Path("/content/decoding_ablation_10") if Path("/content").is_dir() else Path("decoding_ablation_10")

contract = json.loads((BASELINE_DIR / "comparison_contract.json").read_text(encoding="utf-8"))
for pilot_dir in (PILOT_01_DIR, PILOT_02_DIR):
    assert json.loads((pilot_dir / "comparison_contract.json").read_text(encoding="utf-8")) == contract
    cfg = json.loads((pilot_dir / "run_config.json").read_text(encoding="utf-8"))
    assert cfg["comparison_contract_sha256"] == digest(contract)

version = contract["dataset_version"]
prepared_dir = PREPARED_ROOT / version
manifest = json.loads((prepared_dir / "manifest.json").read_text(encoding="utf-8"))
prepared = load_from_disk(str(prepared_dir / "dataset"))
for split, sha in manifest["split_sha256"].items():
    assert digest(prepared[split].to_list()) == sha, f"Prepared {split} content changed"
cases = prepared["validation"].to_list()
assert [c["case_id"] for c in cases] == contract["case_ids"]
assert digest([c["prompt_text"] for c in cases]) == contract["prompt_sha256"]

for package in ("transformers", "torch"):
    actual = torch.__version__ if package == "torch" else importlib.metadata.version(package)
    assert actual == contract[package], f"{package} differs from baseline: {actual} vs {contract[package]}"
assert contract["do_sample"] is False and contract["tokenizer_add_special_tokens"] is False
assert contract["metrics_version"] == 1

MAX_NEW_TOKENS, SEED = contract["max_new_tokens"], contract["seed"]
MODEL_ID, model_revision = contract["model"], contract["base_model_revision"]
dtype = torch.float32 if contract["dtype"] == "float32" else torch.bfloat16
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Runtime verified: device={device}, dtype={dtype}, transformers={importlib.metadata.version('transformers')}, torch={torch.__version__}, cases={len(cases)}", flush=True)
if device != "cuda":
    print("WARNING: Running on CPU will be slow; switch Colab runtime to T4 GPU if possible.", flush=True)

Extracting /content/colab_ablation_bundle.zip to /content ...


Extraction complete.


Runtime verified: device=cuda, dtype=torch.float32, transformers=4.57.6, torch=2.14.1+cu130, cases=10


In [ ]:
TOP_LEVEL_KEYS = (
    "visit motivation", "admission", "patient information", "patient medical history",
    "surgeries", "symptoms", "medical examinations", "diagnosis tests", "treatments", "discharge")
FORMAT_INSTRUCTION = (
    "Extract the clinical facts stated in the note into one JSON object. "
    "Return JSON only, without markdown or commentary. Use these top-level fields: "
    + ", ".join(TOP_LEVEL_KEYS) + ". "
    "Use the string None only for information absent from the note. "
    "Do not invent facts or repeat identical entries.")

def no_duplicate_keys(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError("Duplicate JSON key: " + key)
        result[key] = value
    return result

def reject_constant(value):
    raise ValueError("Nonstandard JSON constant: " + value)

def load_object(text, strict=True):
    obj = json.loads(text, strict=strict, object_pairs_hook=no_duplicate_keys,
                     parse_constant=reject_constant)
    if not isinstance(obj, dict):
        raise ValueError("Expected a JSON object")
    return obj

def clean_reference(raw):
    text = raw.strip()
    fenced = text.startswith("```") and text.endswith("```")
    if fenced:
        text = re.sub(r"^```(?:json)?\s*", "", text, count=1, flags=re.IGNORECASE)[:-3].strip()
    try:
        obj = load_object(text)
        repair = "removed_markdown_fence" if fenced else "none"
    except ValueError:
        # Only unescaped control characters are tolerated; serialization escapes them.
        # No guessed commas, missing braces, invented values, or LLM repair.
        obj = load_object(text, strict=False)
        repair = "escaped_control_characters" + ("_and_removed_fence" if fenced else "")
    if set(obj) != set(TOP_LEVEL_KEYS):
        raise ValueError("Unexpected top-level schema")
    canonical = json.dumps(obj, ensure_ascii=False, separators=(",", ":"))
    assert load_object(canonical) == obj
    return canonical, repair

def strip_fences(text):
    text = text.strip()
    if text.startswith("```") and text.endswith("```"):
        return re.sub(r"^```(?:json)?\s*", "", text, count=1, flags=re.IGNORECASE)[:-3].strip()
    return text

def facts(value, path=""):
    """Multiset of (field path, normalized value); array order is ignored."""
    result = Counter()
    if isinstance(value, dict):
        for key, child in value.items():
            result.update(facts(child, f"{path}.{key}" if path else key))
    elif isinstance(value, list):
        for child in value:
            result.update(facts(child, path))
    elif value is not None:
        normalized = " ".join(str(value).casefold().split())
        if normalized not in ("", "none", "null", "unknown", "not provided"):
            result[(path, normalized)] += 1
    return result

def key_paths(value, prefix=""):
    result = set()
    if isinstance(value, dict):
        for key, child in value.items():
            path = f"{prefix}.{key}" if prefix else key
            result.add(path)
            result.update(key_paths(child, path))
    elif isinstance(value, list):
        for child in value:
            result.update(key_paths(child, prefix))
    return result

def evaluate_output(text, reference):
    expected = load_object(reference)
    raw_valid = True
    try:
        load_object(text)
    except ValueError:
        raw_valid = False
    try:
        actual = load_object(strip_fences(text))
        error = None
    except ValueError as exc:
        actual, error = None, str(exc)
    expected_facts = facts(expected)
    actual_facts = facts(actual)
    matches = sum((expected_facts & actual_facts).values())
    n_actual, n_expected = sum(actual_facts.values()), sum(expected_facts.values())
    precision = matches / n_actual if n_actual else 0.0
    recall = matches / n_expected if n_expected else 0.0
    return {"empty": not text.strip(), "raw_json_object_valid": raw_valid,
            "json_object_valid_after_fence_removal": actual is not None,
            "parse_error": error, "exact_key_match": actual is not None and key_paths(actual) == key_paths(expected),
            "nonmissing_generated_values": n_actual, "reference_values": n_expected,
            "matched_values": matches, "placeholder_only": actual is not None and n_actual == 0,
            "value_precision": precision, "value_recall": recall,
            "value_f1": 2*precision*recall/(precision+recall) if precision+recall else 0.0}

ref = '{"age":"29", "symptoms":[{"name":"pain"}]}'
assert evaluate_output('```json\n'+ref+'\n```', ref)["value_f1"] == 1
assert not evaluate_output('```json\n'+ref+'\n```', ref)["raw_json_object_valid"]
assert evaluate_output('{"age":"None"}', ref)["placeholder_only"]
assert evaluate_output('', ref)["value_recall"] == 0
assert evaluate_output('{"age":"29","age":"30"}', ref)["parse_error"]
assert evaluate_output('{"age":"29", "symptoms":[{"name":"pain"},{"name":"pain"}]}', ref)["value_precision"] < 1
print("Metric self-checks passed.")


def count_duplicate_array_items(text):
    try:
        obj = load_object(strip_fences(text))
    except ValueError:
        return None
    dups = 0
    def walk(node):
        nonlocal dups
        if isinstance(node, list):
            seen = set()
            for item in node:
                key = json.dumps(item, ensure_ascii=False, sort_keys=True)
                if key in seen:
                    dups += 1
                seen.add(key)
            for item in node:
                walk(item)
        elif isinstance(node, dict):
            for v in node.values():
                walk(v)
    walk(obj)
    return dups

def max_word_5gram_repeat(text):
    words = re.findall(r"\S+", text.lower())
    if len(words) < 5:
        return 0
    counts = Counter(tuple(words[i:i+5]) for i in range(len(words) - 4))
    return max(counts.values()) if counts else 0

Metric self-checks passed.


In [ ]:
# Explicit Phase-1 ablation protocol (new output directory, preserves baseline & pilots)
DECODING_CONDITIONS = [
    {"tag": "gen_rp_1.08", "repetition_penalty": 1.08, "generated_only": True},
    {"tag": "gen_rp_1.15", "repetition_penalty": 1.15, "generated_only": True},
]
MODEL_SPECS = [
    {"name": "base", "adapter_dir": None},
    {"name": "pilot_01", "adapter_dir": PILOT_01_DIR / "adapter"},
    {"name": "pilot_02", "adapter_dir": PILOT_02_DIR / "adapter"},
]

class GeneratedOnlyRepetitionPenalty(LogitsProcessor):
    """Applies standard repetition_penalty only to tokens generated after prompt_length."""
    def __init__(self, penalty: float, prompt_length: int):
        if penalty <= 0:
            raise ValueError("penalty must be > 0")
        self.penalty = float(penalty)
        self.prompt_length = int(prompt_length)

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        if self.penalty == 1.0 or input_ids.shape[1] <= self.prompt_length:
            return scores
        gen_ids = input_ids[:, self.prompt_length :]
        if gen_ids.shape[0] == 1:
            gen_ids = torch.unique(gen_ids[0]).unsqueeze(0)
        score = torch.gather(scores, 1, gen_ids)
        score = torch.where(score < 0, score * self.penalty, score / self.penalty)
        return scores.scatter(1, gen_ids, score)

ablation_config = {
    "protocol_name": "decoding_ablation_v1",
    "comparison_contract_sha256": digest(contract),
    "dataset_version": version,
    "metric_source_sha256": "021d67e1183e4c5be0f1af69cedcbb73749e0388fc27033b305a25c86640790e",
    "decoding_conditions": DECODING_CONDITIONS,
    "models": [m["name"] for m in MODEL_SPECS],
    "peft": importlib.metadata.version("peft"),
}
ablation_dir = OUTPUT_ROOT / (version + "_" + digest(ablation_config)[:12])
ablation_dir.mkdir(parents=True, exist_ok=True)
cfg_path = ablation_dir / "ablation_config.json"
if cfg_path.exists():
    assert json.loads(cfg_path.read_text(encoding="utf-8")) == ablation_config
else:
    cfg_path.write_text(json.dumps(ablation_config, indent=2), encoding="utf-8")
(ablation_dir / "comparison_contract.json").write_text(json.dumps(contract, indent=2), encoding="utf-8")

token = read_token()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=model_revision, token=token)
pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id

for spec in MODEL_SPECS:
    model_name = spec["name"]
    random.seed(SEED)
    torch.manual_seed(SEED)
    print(f"\n=== Loading model: {model_name} ===", flush=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, revision=model_revision, token=token, dtype=dtype
    ).to(device)
    if spec["adapter_dir"] is not None:
        model = PeftModel.from_pretrained(base_model, str(spec["adapter_dir"]), is_trainable=False).eval()
    else:
        model = base_model.eval()
    eos = model.generation_config.eos_token_id
    eos_ids = eos if isinstance(eos, list) else [eos]

    for cond in DECODING_CONDITIONS:
        run_dir = ablation_dir / f"{model_name}__{cond['tag']}"
        run_dir.mkdir(parents=True, exist_ok=True)
        gen_path = run_dir / "generations.json"
        diag_path = run_dir / "generation_diagnostics.json"
        generations = json.loads(gen_path.read_text(encoding="utf-8")) if gen_path.exists() else {}
        diagnostics = json.loads(diag_path.read_text(encoding="utf-8")) if diag_path.exists() else {}
        started = time.monotonic()
        for case in cases:
            key = case["case_id"]
            if generations.get(key, "").strip() and key in diagnostics:
                continue
            inputs = tokenizer(case["prompt_text"], return_tensors="pt", add_special_tokens=False).to(device)
            n_prompt = inputs.input_ids.shape[1]
            assert n_prompt + MAX_NEW_TOKENS <= model.config.max_position_embeddings
            with torch.inference_mode():
                logits = model(**inputs, logits_to_keep=1).logits[:, -1, :]
                finite = bool(torch.isfinite(logits).all().item())
                del logits
                if not finite:
                    ids = []
                else:
                    gen_kwargs = {
                        "max_new_tokens": MAX_NEW_TOKENS,
                        "do_sample": False,
                        "pad_token_id": pad_id,
                    }
                    if cond["generated_only"] and cond["repetition_penalty"] != 1.0:
                        gen_kwargs["logits_processor"] = LogitsProcessorList([
                            GeneratedOnlyRepetitionPenalty(cond["repetition_penalty"], n_prompt)
                        ])
                    elif cond["repetition_penalty"] != 1.0:
                        gen_kwargs["repetition_penalty"] = cond["repetition_penalty"]
                    generated = model.generate(**inputs, **gen_kwargs)
                    ids = generated[0, n_prompt:].tolist()
            text = tokenizer.decode(ids, skip_special_tokens=True)
            diagnostics[key] = {
                "prompt_tokens": n_prompt,
                "generated_tokens": len(ids),
                "finite_first_token_logits": finite,
                "empty_output": not text.strip(),
                "reached_budget": len(ids) >= MAX_NEW_TOKENS,
                "ended_with_eos": bool(ids) and ids[-1] in eos_ids,
                "token_ids": ids,
            }
            diag_path.write_text(json.dumps(diagnostics, ensure_ascii=False, indent=2), encoding="utf-8")
            if not finite or not text.strip():
                raise RuntimeError(f"{model_name}/{cond['tag']}/{key}: failed generation")
            generations[key] = text
            gen_path.write_text(json.dumps(generations, ensure_ascii=False, indent=2), encoding="utf-8")
            print(f"[{model_name} | {cond['tag']}] {len(generations)}/{len(cases)} {key}: {len(ids)} toks ({time.monotonic()-started:.1f}s)", flush=True)

    del model, base_model
    if device == "cuda":
        torch.cuda.empty_cache()

print("\nAll decoding ablation generations complete:", ablation_dir, flush=True)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

... (65 outputs left out)

[pilot_02 | gen_rp_1.15] 8/10 validation:428: 818 toks (317.5s)


[pilot_02 | gen_rp_1.15] 9/10 validation:2926: 859 toks (363.7s)


[pilot_02 | gen_rp_1.15] 10/10 validation:2336: 565 toks (393.3s)



All decoding ablation generations complete: /content/decoding_ablation_10/900825a06d38a006_0c93337f20a5


In [ ]:
def score_ablation_run(outputs, diagnostics):
    records = []
    for case in cases:
        key = case["case_id"]
        text = outputs[key]
        metrics = evaluate_output(text, case["reference"])
        records.append({
            "case_id": key,
            **metrics,
            "reached_budget": bool(diagnostics[key].get("reached_budget")),
            "generated_tokens": int(diagnostics[key]["generated_tokens"]),
            "duplicate_array_items": count_duplicate_array_items(text),
            "max_5gram_repeat": max_word_5gram_repeat(text),
            "generated": text,
        })
    frame = pd.DataFrame(records)
    summary = {"selected_cases": len(cases), "evaluated_cases": len(records)}
    for name in ("empty", "raw_json_object_valid", "json_object_valid_after_fence_removal",
                 "exact_key_match", "placeholder_only", "reached_budget"):
        summary[name + "_cases"] = int(frame[name].sum())
    for name in ("value_precision", "value_recall", "value_f1"):
        summary["mean_" + name] = float(frame[name].mean())
    for key, col in (("matched_values", "matched_values"), ("reference_values", "reference_values"),
                     ("generated_values", "nonmissing_generated_values")):
        summary[key] = int(frame[col].sum())
    summary["mean_generated_tokens"] = float(frame["generated_tokens"].mean())
    summary["max_5gram_repeat_ge_3_cases"] = int((frame["max_5gram_repeat"] >= 3).sum())
    return frame, summary

summary_rows = []

# Include baseline rp=1.00 directly from frozen baseline artifacts
base_gens = json.loads((BASELINE_DIR / "generations.json").read_text(encoding="utf-8"))
base_diags = json.loads((BASELINE_DIR / "generation_diagnostics.json").read_text(encoding="utf-8"))
_, base_10_sum = score_ablation_run(base_gens, base_diags)
summary_rows.append({"model": "base", "decoding": "rp_1.00", **base_10_sum})

# Include pilot_01 & pilot_02 rp=1.00 if their comparison folders are present in runtime
for p_name, p_glob in (("pilot_01", "pilot_01_comparision/*/finetuned_per_case.csv"),
                       ("pilot_02", "pilot_02_comparision/*/finetuned_per_case.csv")):
    matches = sorted(PREPARED_ROOT.parent.glob(p_glob))
    if matches:
        p_df = pd.read_csv(matches[0])
        p_gens = {row["case_id"]: (str(row["generated"]) if pd.notna(row["generated"]) else "") for _, row in p_df.iterrows()}
        p_diags = {row["case_id"]: {"reached_budget": bool(row["reached_budget"]), "generated_tokens": int(row["generated_tokens"])} for _, row in p_df.iterrows()}
        _, p_sum = score_ablation_run(p_gens, p_diags)
        summary_rows.append({"model": p_name, "decoding": "rp_1.00", **p_sum})

for spec in MODEL_SPECS:
    for cond in DECODING_CONDITIONS:
        run_dir = ablation_dir / f"{spec['name']}__{cond['tag']}"
        gens = json.loads((run_dir / "generations.json").read_text(encoding="utf-8"))
        diags = json.loads((run_dir / "generation_diagnostics.json").read_text(encoding="utf-8"))
        frame, run_summary = score_ablation_run(gens, diags)
        frame.to_csv(run_dir / "per_case.csv", index=False)
        (run_dir / "aggregate.json").write_text(json.dumps(run_summary, indent=2), encoding="utf-8")
        summary_rows.append({"model": spec["name"], "decoding": cond["tag"], **run_summary})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(ablation_dir / "ablation_summary.csv", index=False)
(ablation_dir / "ablation_summary.json").write_text(json.dumps(summary_rows, indent=2), encoding="utf-8")
display(summary_df)
print("Saved ablation summary to:", (ablation_dir / "ablation_summary.csv").resolve())

,model,decoding,selected_cases,evaluated_cases,empty_cases,raw_json_object_valid_cases,json_object_valid_after_fence_removal_cases,exact_key_match_cases,placeholder_only_cases,reached_budget_cases,mean_value_precision,mean_value_recall,mean_value_f1,matched_values,reference_values,generated_values,mean_generated_tokens,max_5gram_repeat_ge_3_cases
0,base,rp_1.00,10,10,0,0,4,0,0,3,0.006667,0.002000,0.003077,1,353,57,1049.1,8
1,pilot_01,rp_1.00,10,10,0,6,6,2,0,1,0.166346,0.096354,0.118740,34,353,124,751.0,6
2,pilot_02,rp_1.00,10,10,0,5,5,2,0,4,0.125493,0.097183,0.103366,33,353,156,1236.4,5
3,base,gen_rp_1.08,10,10,0,0,6,0,0,1,0.029455,0.025739,0.025332,7,353,163,831.6,8
4,base,gen_rp_1.15,10,10,0,0,7,0,0,0,0.014045,0.010696,0.011574,3,353,237,798.3,4
5,pilot_01,gen_rp_1.08,10,10,0,6,6,1,0,0,0.132063,0.111810,0.117388,32,353,162,688.7,4
6,pilot_01,gen_rp_1.15,10,10,0,2,2,0,0,2,0.038225,0.029739,0.032235,9,353,47,920.3,3
7,pilot_02,gen_rp_1.08,10,10,0,9,9,3,0,0,0.188898,0.180838,0.179802,58,353,283,712.8,4
8,pilot_02,gen_rp_1.15,10,10,0,6,6,0,0,0,0.085453,0.079709,0.080034,28,353,212,738.3,6


Saved ablation summary to: /content/decoding_ablation_10/900825a06d38a006_0c93337f20a5/ablation_summary.csv


In [ ]:
import shutil
archive = shutil.make_archive(str(ablation_dir.resolve()), "zip", root_dir=ablation_dir.parent, base_dir=ablation_dir.name)
print("Archive ready for download:", archive)

Archive ready for download: /content/decoding_ablation_10/900825a06d38a006_0c93337f20a5.zip
